In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
import numpy as np
from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model import ForagerCollection


from aind_dynamic_foraging_models.generative_model.foragers import ForagerCollection


forager_collection: ForagerCollection = ForagerCollection()
df = forager_collection.get_all_foragers()

df


In [ ]:
from __future__ import annotations

import copy
import itertools
import json
import re
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import cloudpickle
import numpy as np
import pandas as pd

from aind_behavior_gym.dynamic_foraging.task import CoupledBlockTask, UncoupledBlockTask
from aind_dynamic_foraging_models.generative_model.foragers import ForagerCollection


# =============================================================================
# Helpers
# =============================================================================

def _to_jsonable(obj: Any) -> Any:
    """Convert nested objects into JSON-serializable objects."""
    if isinstance(obj, dict):
        return {str(k): _to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.bool_,)):
        return bool(obj)
    return obj


def _sanitize_name(text: str) -> str:
    """Sanitize a string for filesystem paths."""
    text = str(text)
    text = re.sub(r"[^A-Za-z0-9._-]+", "_", text)
    text = re.sub(r"_+", "_", text).strip("_")
    return text


def _save_cloudpickle(obj: Any, path: Path) -> None:
    """Save an object with cloudpickle."""
    with path.open("wb") as f:
        cloudpickle.dump(obj, f)


def _save_json(data: Dict[str, Any], path: Path) -> None:
    """Save a JSON file."""
    with path.open("w") as f:
        json.dump(_to_jsonable(data), f, indent=2)


def _expand_param_grid(param_grid: Optional[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Expand a parameter grid into all combinations.

    Scalars are treated as length-1 lists.
    """
    if not param_grid:
        return [{}]

    keys = list(param_grid.keys())
    values_list: List[List[Any]] = []

    for k in keys:
        v = param_grid[k]
        if isinstance(v, np.ndarray):
            values = v.tolist()
        elif isinstance(v, (list, tuple)):
            values = list(v)
        else:
            values = [v]
        values_list.append(values)

    combos: List[Dict[str, Any]] = []
    for combo_vals in itertools.product(*values_list):
        combos.append(dict(zip(keys, combo_vals)))
    return combos


def _filter_valid_task_param_combos(task_param_combos: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Keep only valid task parameter combinations.

    Currently filters out combinations where block_max < block_min.
    """
    valid_combos: List[Dict[str, Any]] = []

    for task_params in task_param_combos:
        block_min = task_params.get("block_min", None)
        block_max = task_params.get("block_max", None)

        if block_min is not None and block_max is not None:
            if block_max < block_min:
                continue

        valid_combos.append(task_params)

    return valid_combos


# =============================================================================
# Forager helpers
# =============================================================================

def _clone_forager_from_df(df: pd.DataFrame, alias: str, seed: int = 42):
    """
    Fetch a fresh forager object from df['agent_alias'] / df['forager'].
    """
    rows = df.loc[df["agent_alias"] == alias]

    if len(rows) == 0:
        raise ValueError(f"Alias not found in df['agent_alias']: {alias}")
    if len(rows) > 1:
        raise ValueError(f"Alias is not unique in df['agent_alias']: {alias}")

    obj = rows.iloc[0]["forager"]

    if hasattr(obj, "fit") and hasattr(obj, "perform"):
        agent = copy.deepcopy(obj)
        if hasattr(agent, "seed"):
            try:
                agent.seed = seed
            except Exception:
                pass
        return agent

    if callable(obj):
        try:
            return obj(seed=seed)
        except TypeError:
            try:
                return obj()
            except Exception as e:
                raise RuntimeError(
                    f"Could not instantiate forager from callable for alias={alias}"
                ) from e

    raise TypeError(
        f"Unsupported object type in df['forager'] for alias={alias}: {type(obj)}"
    )


def _set_params_if_requested(agent, params_override: Optional[Dict[str, Any]]) -> None:
    """Set agent parameters if an override dict is provided."""
    if not params_override:
        return

    if not hasattr(agent, "set_params"):
        raise AttributeError("Agent does not have set_params(...)")

    agent.set_params(**params_override)


# =============================================================================
# Task helpers
# =============================================================================

def _build_task(
    *,
    use_uncoupled: bool,
    seed: int,
    task_params: Dict[str, Any],
):
    """
    Build a dynamic-foraging task from a parameter dict.

    All task parameters, including reward_baiting, should be passed in task_params.
    """
    params = dict(task_params)
    params["seed"] = seed

    if use_uncoupled:
        return UncoupledBlockTask(**params)
    return CoupledBlockTask(**params)


def _validate_task_combo(task_params: Dict[str, Any]) -> None:
    """Perform basic validation for one task parameter combination."""
    block_min = task_params.get("block_min", None)
    block_max = task_params.get("block_max", None)

    if block_min is not None and block_max is not None and block_min > block_max:
        raise ValueError(
            f"Invalid task combo: block_min ({block_min}) > block_max ({block_max})"
        )


def _build_protocol_name(*, use_uncoupled: bool, task_params: Dict[str, Any]) -> str:
    """Return protocol name from task family and reward_baiting."""
    task_name = "Uncoupled" if use_uncoupled else "Coupled"
    baiting_name = "Baiting" if bool(task_params.get("reward_baiting", False)) else "NoBaiting"
    return f"{task_name} {baiting_name}"


# =============================================================================
# Extraction helpers
# =============================================================================

def _extract_task_reward_array(task: Any) -> np.ndarray:
    """
    Extract realized reward array from task if available.

    Expected to be a 1D array-like of 0/1 values per trial.
    """
    reward = getattr(task, "reward", None)
    if reward is None:
        return np.array([], dtype=float)

    arr = np.asarray(reward)
    if arr.ndim == 0:
        return np.array([float(arr)])
    if arr.ndim > 1:
        arr = arr.reshape(-1)

    return arr.astype(float, copy=False)


def _extract_agent_choice_history(agent: Any) -> np.ndarray:
    """Extract choice history from the agent after perform(task)."""
    if hasattr(agent, "get_choice_history"):
        try:
            return np.asarray(agent.get_choice_history())
        except Exception:
            pass

    if hasattr(agent, "choice_history"):
        try:
            return np.asarray(agent.choice_history)
        except Exception:
            pass

    return np.array([], dtype=float)


def _extract_agent_reward_history(agent: Any) -> np.ndarray:
    """Extract reward history from the agent after perform(task)."""
    if hasattr(agent, "get_reward_history"):
        try:
            return np.asarray(agent.get_reward_history())
        except Exception:
            pass

    if hasattr(agent, "reward_history"):
        try:
            return np.asarray(agent.reward_history)
        except Exception:
            pass

    return np.array([], dtype=float)


# =============================================================================
# Summary helpers
# =============================================================================

def _safe_numeric_stats(arr: np.ndarray, prefix: str) -> Dict[str, Any]:
    """Compute simple numeric summary statistics for an array."""
    if arr.size == 0:
        return {
            f"{prefix}_n": 0,
            f"{prefix}_sum": np.nan,
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
        }

    valid = np.isfinite(arr)
    arr = arr[valid]
    if arr.size == 0:
        return {
            f"{prefix}_n": 0,
            f"{prefix}_sum": np.nan,
            f"{prefix}_mean": np.nan,
            f"{prefix}_std": np.nan,
        }

    return {
        f"{prefix}_n": int(arr.size),
        f"{prefix}_sum": float(np.sum(arr)),
        f"{prefix}_mean": float(np.mean(arr)),
        f"{prefix}_std": float(np.std(arr, ddof=0)),
    }


def _summarize_choices(choice_history: np.ndarray) -> Dict[str, Any]:
    """
    Summarize choice behavior.

    Assumes:
    - left choice is encoded as 0
    - right choice is encoded as 1
    - ignored / invalid values may be something else or NaN
    """
    if choice_history.size == 0:
        return dict(
            n_trials_observed=0,
            n_choice_left=0,
            n_choice_right=0,
            n_choice_other=0,
            p_choose_left=np.nan,
            p_choose_right=np.nan,
        )

    valid = np.isfinite(choice_history)
    ch = choice_history[valid]

    n_left = int(np.sum(ch == 0))
    n_right = int(np.sum(ch == 1))
    n_other = int(len(ch) - n_left - n_right)
    n_total = int(len(ch))

    return dict(
        n_trials_observed=n_total,
        n_choice_left=n_left,
        n_choice_right=n_right,
        n_choice_other=n_other,
        p_choose_left=float(n_left / n_total) if n_total > 0 else np.nan,
        p_choose_right=float(n_right / n_total) if n_total > 0 else np.nan,
    )


def _summarize_rewards(task_reward: np.ndarray, agent_reward_history: np.ndarray) -> Dict[str, Any]:
    """Summarize reward outcomes from both task.reward and agent reward history."""
    out: Dict[str, Any] = {}
    out.update(_safe_numeric_stats(task_reward, "task_reward"))
    out.update(_safe_numeric_stats(agent_reward_history, "agent_reward"))
    return out


# =============================================================================
# Directory helpers
# =============================================================================

def _build_run_dir(
    *,
    output_root: Path,
    task_combo_id: int,
    forager_alias: str,
    forager_param_combo_id: int,
    repeat_idx: int,
) -> Path:
    """Create a stable output directory for one task x forager-param x repeat run."""
    run_dir = (
        output_root
        / f"task_combo_{task_combo_id:05d}"
        / _sanitize_name(forager_alias)
        / f"forager_combo_{forager_param_combo_id:05d}"
        / f"repeat_{repeat_idx:04d}"
    )
    run_dir.mkdir(parents=True, exist_ok=True)
    return run_dir


# =============================================================================
# Worker
# =============================================================================

def _worker_run_single_task_performance(
    *,
    task_combo_id: int,
    task_params: Dict[str, Any],
    repeat_idx: int,
    output_root: str,
    forager_alias: str,
    forager_param_combo_id: int,
    forager_params: Optional[Dict[str, Any]],
    use_uncoupled_task: bool,
    task_seed: int,
    model_seed: int,
    override_existing: bool,
) -> Dict[str, Any]:
    """
    One worker unit:
    - build one task
    - instantiate one agent
    - optionally set one forager parameter combination
    - run agent.perform(task)
    - save task and agent
    - save summary.json
    - return one summary row
    """
    output_root = Path(output_root)

    run_dir = _build_run_dir(
        output_root=output_root,
        task_combo_id=task_combo_id,
        forager_alias=forager_alias,
        forager_param_combo_id=forager_param_combo_id,
        repeat_idx=repeat_idx,
    )

    summary_path = run_dir / "summary.json"
    if summary_path.exists() and not override_existing:
        with summary_path.open("r") as f:
            return json.load(f)

    _validate_task_combo(task_params)

    forager_collection = ForagerCollection()
    df = forager_collection.get_all_foragers()

    if "agent_alias" not in df.columns or "forager" not in df.columns:
        raise ValueError(
            "Expected df from get_all_foragers() to contain columns: "
            "'agent_alias' and 'forager'"
        )

    task = _build_task(
        use_uncoupled=use_uncoupled_task,
        seed=task_seed,
        task_params=task_params,
    )

    agent = _clone_forager_from_df(df, forager_alias, seed=model_seed)
    _set_params_if_requested(agent, forager_params)

    agent.perform(task)

    task_reward = _extract_task_reward_array(task)
    choice_history = _extract_agent_choice_history(agent)
    reward_history = _extract_agent_reward_history(agent)

    choice_summary = _summarize_choices(choice_history)
    reward_summary = _summarize_rewards(task_reward, reward_history)

    num_trials = task_params.get("num_trials", None)
    if num_trials is None and len(choice_history) > 0:
        num_trials = int(len(choice_history))

    summary: Dict[str, Any] = dict(
        status="ok",
        kind="task_performance",
        protocol=_build_protocol_name(
            use_uncoupled=use_uncoupled_task,
            task_params=task_params,
        ),
        use_uncoupled_task=bool(use_uncoupled_task),
        task_combo_id=int(task_combo_id),
        task_params=task_params,
        reward_baiting=bool(task_params.get("reward_baiting", False)),
        repeat_idx=int(repeat_idx),
        task_seed=int(task_seed),
        model_seed=int(model_seed),
        forager_alias=forager_alias,
        forager_param_combo_id=int(forager_param_combo_id),
        forager_params=forager_params if forager_params is not None else {},
        num_trials=int(num_trials) if num_trials is not None else None,
        output_dir=str(run_dir),
        task_pickle_file="task.cloudpickle",
        agent_pickle_file="performed_agent.cloudpickle",
    )

    summary.update(choice_summary)
    summary.update(reward_summary)

    _save_cloudpickle(task, run_dir / "task.cloudpickle")
    _save_cloudpickle(agent, run_dir / "performed_agent.cloudpickle")
    _save_json(summary, summary_path)

    return summary


# =============================================================================
# Aggregate summary writers
# =============================================================================

def _write_forager_combo_summary(combo_dir: Path, override_existing: bool) -> None:
    """
    Aggregate repeat-level summaries inside one forager_combo directory.
    """
    out_file = combo_dir / "forager_combo_summary.json"

    if out_file.exists() and not override_existing:
        return

    rows: List[Dict[str, Any]] = []

    for summary_file in sorted(combo_dir.glob("repeat_*/summary.json")):
        with summary_file.open("r") as f:
            rows.append(json.load(f))

    if len(rows) == 0:
        return

    df = pd.DataFrame(rows)

    task_reward_mean_series = pd.to_numeric(df.get("task_reward_mean"), errors="coerce")
    agent_reward_mean_series = pd.to_numeric(df.get("agent_reward_mean"), errors="coerce")
    agent_reward_sum_series = pd.to_numeric(df.get("agent_reward_sum"), errors="coerce")

    summary = dict(
        status="ok",
        kind="forager_combo_summary",
        task_combo_id=int(rows[0]["task_combo_id"]),
        task_params=rows[0]["task_params"],
        reward_baiting=bool(rows[0].get("reward_baiting", False)),
        forager_alias=rows[0]["forager_alias"],
        forager_param_combo_id=int(rows[0]["forager_param_combo_id"]),
        forager_params=rows[0]["forager_params"],
        n_repeats=int(len(df)),
        mean_task_reward_mean=float(task_reward_mean_series.mean()),
        std_task_reward_mean=float(task_reward_mean_series.std(ddof=0)),
        mean_agent_reward_mean=float(agent_reward_mean_series.mean()),
        std_agent_reward_mean=float(agent_reward_mean_series.std(ddof=0)),
        mean_agent_reward_sum=float(agent_reward_sum_series.mean()),
        std_agent_reward_sum=float(agent_reward_sum_series.std(ddof=0)),
    )

    _save_json(summary, out_file)


def _write_task_combo_summary(task_dir: Path, override_existing: bool) -> None:
    """
    Aggregate results across all models and parameter combos for one task_combo.
    """
    out_file = task_dir / "task_combo_summary.json"

    if out_file.exists() and not override_existing:
        return

    rows: List[Dict[str, Any]] = []

    for summary_file in sorted(task_dir.glob("*/*/repeat_*/summary.json")):
        with summary_file.open("r") as f:
            rows.append(json.load(f))

    if len(rows) == 0:
        return

    df = pd.DataFrame(rows)
    agent_reward_mean_series = pd.to_numeric(df.get("agent_reward_mean"), errors="coerce")
    agent_reward_sum_series = pd.to_numeric(df.get("agent_reward_sum"), errors="coerce")

    model_summary: Dict[str, Any] = {}

    for model in sorted(df["forager_alias"].unique()):
        sub = df[df["forager_alias"] == model].copy()

        sub_agent_reward_mean = pd.to_numeric(sub.get("agent_reward_mean"), errors="coerce")
        sub_agent_reward_sum = pd.to_numeric(sub.get("agent_reward_sum"), errors="coerce")

        best_idx = sub_agent_reward_mean.idxmax()
        best_row = sub.loc[best_idx].to_dict() if len(sub) > 0 else None

        model_summary[model] = dict(
            n_runs=int(len(sub)),
            mean_agent_reward_mean=float(sub_agent_reward_mean.mean()),
            std_agent_reward_mean=float(sub_agent_reward_mean.std(ddof=0)),
            mean_agent_reward_sum=float(sub_agent_reward_sum.mean()),
            std_agent_reward_sum=float(sub_agent_reward_sum.std(ddof=0)),
            best_forager_param_combo_id=int(best_row["forager_param_combo_id"]) if best_row is not None else None,
            best_forager_params=best_row["forager_params"] if best_row is not None else None,
            best_agent_reward_mean=float(best_row["agent_reward_mean"]) if best_row is not None else None,
            best_agent_reward_sum=float(best_row["agent_reward_sum"]) if best_row is not None else None,
        )

    global_best_idx = agent_reward_mean_series.idxmax()
    global_best_row = df.loc[global_best_idx].to_dict()

    summary = dict(
        status="ok",
        kind="task_combo_summary",
        task_combo_id=int(rows[0]["task_combo_id"]),
        task_params=rows[0]["task_params"],
        reward_baiting=bool(rows[0].get("reward_baiting", False)),
        n_runs=int(len(df)),
        mean_agent_reward_mean=float(agent_reward_mean_series.mean()),
        std_agent_reward_mean=float(agent_reward_mean_series.std(ddof=0)),
        mean_agent_reward_sum=float(agent_reward_sum_series.mean()),
        std_agent_reward_sum=float(agent_reward_sum_series.std(ddof=0)),
        best_forager_alias=global_best_row["forager_alias"],
        best_forager_param_combo_id=int(global_best_row["forager_param_combo_id"]),
        best_forager_params=global_best_row["forager_params"],
        best_agent_reward_mean=float(global_best_row["agent_reward_mean"]),
        best_agent_reward_sum=float(global_best_row["agent_reward_sum"]),
        models=model_summary,
    )

    _save_json(summary, out_file)


# =============================================================================
# Main runner
# =============================================================================

def run_task_performance_grid_parallel(
    *,
    output_root: Path,
    forager_aliases: Sequence[str],
    task_param_grid: Dict[str, Any],
    n_repeats: int,
    random_seed_base: int,
    task_seed_base: int,
    use_uncoupled_task: bool = True,
    forager_param_grid_by_alias: Optional[Dict[str, Dict[str, Any]]] = None,
    n_jobs: int = 1,
    override_existing: bool = False,
) -> pd.DataFrame:
    """
    Compare model performance across:
    - task parameter combinations
    - forager parameter combinations
    - repeats

    Returns one row per run.
    """
    output_root.mkdir(parents=True, exist_ok=True)

    raw_task_param_combos = _expand_param_grid(task_param_grid)
    task_param_combos = _filter_valid_task_param_combos(raw_task_param_combos)

    forager_param_grid_by_alias = forager_param_grid_by_alias or {}

    jobs: List[Dict[str, Any]] = []

    for task_combo_id, task_params in enumerate(task_param_combos):
        for repeat_idx in range(n_repeats):
            task_seed = task_seed_base + task_combo_id * 100000 + repeat_idx

            for agent_idx, forager_alias in enumerate(forager_aliases):
                forager_param_grid = forager_param_grid_by_alias.get(forager_alias, {})
                forager_param_combos = _expand_param_grid(forager_param_grid)

                for forager_param_combo_id, forager_params in enumerate(forager_param_combos):
                    model_seed = (
                        random_seed_base
                        + task_combo_id * 1000000
                        + repeat_idx * 10000
                        + agent_idx * 1000
                        + forager_param_combo_id
                    )

                    jobs.append(
                        dict(
                            task_combo_id=task_combo_id,
                            task_params=task_params,
                            repeat_idx=repeat_idx,
                            output_root=str(output_root),
                            forager_alias=forager_alias,
                            forager_param_combo_id=forager_param_combo_id,
                            forager_params=forager_params,
                            use_uncoupled_task=use_uncoupled_task,
                            task_seed=task_seed,
                            model_seed=model_seed,
                            override_existing=override_existing,
                        )
                    )

    rows: List[Dict[str, Any]] = []
    failures: List[Dict[str, Any]] = []

    total = len(jobs)
    print(f"0/{total}")

    with ProcessPoolExecutor(max_workers=n_jobs) as ex:
        future_to_job = {
            ex.submit(_worker_run_single_task_performance, **job): job
            for job in jobs
        }

        finished = 0
        for fut in as_completed(future_to_job):
            job = future_to_job[fut]
            finished += 1

            try:
                rows.append(fut.result())
            except Exception as e:
                failures.append(
                    dict(
                        task_combo_id=job["task_combo_id"],
                        repeat_idx=job["repeat_idx"],
                        forager_alias=job["forager_alias"],
                        forager_param_combo_id=job["forager_param_combo_id"],
                        task_seed=job["task_seed"],
                        model_seed=job["model_seed"],
                        task_params=_to_jsonable(job["task_params"]),
                        forager_params=_to_jsonable(job["forager_params"]),
                        error=repr(e),
                    )
                )

            print(f"{finished}/{total}")

    summary_df = pd.DataFrame(rows)
    if len(summary_df) > 0:
        summary_df = summary_df.sort_values(
            ["task_combo_id", "forager_alias", "forager_param_combo_id", "repeat_idx"]
        )

    summary_df.to_csv(output_root / "all_task_performance_summary.csv", index=False)
    pd.DataFrame(failures).to_csv(output_root / "failures.csv", index=False)

    # Write per-forager-combo and per-task-combo summary JSON files.
    for task_dir in sorted(output_root.glob("task_combo_*")):
        for combo_dir in sorted(task_dir.glob("*/*")):
            if combo_dir.is_dir():
                _write_forager_combo_summary(combo_dir, override_existing=override_existing)
        _write_task_combo_summary(task_dir, override_existing=override_existing)

    run_config = dict(
        output_root=str(output_root),
        forager_aliases=list(forager_aliases),
        task_param_grid=_to_jsonable(task_param_grid),
        n_raw_task_param_combinations=int(len(raw_task_param_combos)),
        n_valid_task_param_combinations=int(len(task_param_combos)),
        forager_param_grid_by_alias=_to_jsonable(forager_param_grid_by_alias),
        n_repeats=int(n_repeats),
        random_seed_base=int(random_seed_base),
        task_seed_base=int(task_seed_base),
        use_uncoupled_task=bool(use_uncoupled_task),
        n_jobs=int(n_jobs),
        override_existing=bool(override_existing),
        n_total_runs=int(len(jobs)),
        n_failures=int(len(failures)),
    )
    _save_json(run_config, output_root / "run_config.json")

    return summary_df



In [ ]:

# =============================================================================
# Example user config
# =============================================================================

OUTPUT_ROOT = Path("/root/capsule/scratch/task_performance_comparison")

RANDOM_SEED_BASE = 42
TASK_SEED_BASE = 53

USE_UNCOUPLED_TASK = True

FORAGER_ALIASES = [
    "QLearning_L1F1_CK1_softmax",
    "QLearning_L2F1_CK1_softmax",
    "ForagingCompareThreshold_L2_CKnone_ResetT_StayBiasT_SideBiasT_FixThrF",
    "ForagingCompareThreshold_L1_CKnone_ResetT_StayBiasT_SideBiasT_FixThrF",
]

# All task parameters, including reward_baiting, are part of the task grid.
# Invalid combinations where block_max < block_min will be filtered out automatically.
TASK_PARAM_GRID: Dict[str, Any] = {
    "rwd_prob_array": [
        [0.1, 0.5, 0.9],
        [0.2, 0.4, 0.7],
    ],
    "block_min": [20, 40],
    "block_max": [35, 60],
    "reward_baiting": [False, True],
    "persev_add": [True],
    "perseverative_limit": [4],
    "max_block_tally": [4],
    "allow_ignore": [True],
    "num_trials": [1000],
}

# Each alias gets its own parameter grid.
# All combinations for each alias will be expanded automatically.
FORAGER_PARAM_GRID_BY_ALIAS: Dict[str, Dict[str, Any]] = {
    "QLearning_L1F1_CK1_softmax": {
        "learn_rate": [0.1, 0.3, 0.6],
        "forget_rate_unchosen": [0.0, 0.2],
        "choice_kernel_relative_weight": [0.0, 0.5],
        "softmax_inverse_temperature": [1.0, 5.0, 10.0],
        "biasL": [-1.0, 0.0, 1.0],
    },
    "QLearning_L2F1_CK1_softmax": {
        "learn_rate_rew": [0.1, 0.3],
        "learn_rate_unrew": [0.1, 0.3],
        "forget_rate_unchosen": [0.0, 0.2],
        "choice_kernel_relative_weight": [0.0, 0.5],
        "softmax_inverse_temperature": [1.0, 5.0],
        "biasL": [-1.0, 0.0, 1.0],
    },
    "ForagingCompareThreshold_L2_CKnone_ResetT_StayBiasT_SideBiasT_FixThrF": {
        "learn_rate_rew": [0.1, 0.3],
        "learn_rate_unrew": [0.1, 0.3],
        "threshold": [-2.0, 0.0, 2.0],
        "stay_bias": [-1.0, 0.0, 1.0],
        "biasL": [-1.0, 0.0, 1.0],
    },
    "ForagingCompareThreshold_L1_CKnone_ResetT_StayBiasT_SideBiasT_FixThrF": {
        "learn_rate": [0.1, 0.3, 0.6],
        "threshold": [-2.0, 0.0, 2.0],
        "stay_bias": [-1.0, 0.0, 1.0],
        "biasL": [-1.0, 0.0, 1.0],
    },
}

N_REPEATS = 20
N_JOBS = 60

# If False:
# - existing repeat-level summary.json is reused
# - existing forager_combo_summary.json is reused
# - existing task_combo_summary.json is reused
#
# If True:
# - all of the above are recomputed and overwritten
OVERRIDE_EXISTING = False


# =============================================================================
# Run
# =============================================================================

if __name__ == "__main__":
    summary_df = run_task_performance_grid_parallel(
        output_root=OUTPUT_ROOT,
        forager_aliases=FORAGER_ALIASES,
        task_param_grid=TASK_PARAM_GRID,
        n_repeats=N_REPEATS,
        random_seed_base=RANDOM_SEED_BASE,
        task_seed_base=TASK_SEED_BASE,
        use_uncoupled_task=USE_UNCOUPLED_TASK,
        forager_param_grid_by_alias=FORAGER_PARAM_GRID_BY_ALIAS,
        n_jobs=N_JOBS,
        override_existing=OVERRIDE_EXISTING,
    )

    print("\n================ TASK PERFORMANCE SUMMARY ================\n")
    if len(summary_df) > 0:
        print(summary_df.head(20).to_string(index=False))
    else:
        print("No successful runs.")